# Lab Day 10 — Finetune Qwen3-VL-4B (QLoRA) trên Kaggle

Notebook hoàn thiện lab theo `GUIDE.md`: **E0** (zero-shot), **E1–E3** (QLoRA), **E4** (ablation), **E5** (tuỳ chọn GRPO).
Trọng tài chấm điểm là `scripts/score.py` — **KHÔNG sửa** `scripts/` và `tests/`.

## Cách chạy trên Kaggle
1. **Settings → Accelerator → GPU T4 x2** (hoặc P100). Bật **Internet ON**.
2. **Add Input → Datasets**: thêm `lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes`.
   Notebook tự tìm trong `/kaggle/input`, không cần Kaggle key. (Nếu muốn tải bằng CLI thì thêm
   **Add-ons → Secrets**: `KAGGLE_USERNAME`, `KAGGLE_KEY`.)
3. Chạy lần lượt các cell. E1–E3 mỗi lần ≈ 20–30 phút (300 bước). Ước lượng này phải ghi lại **thời gian thật**.
4. Kết quả sinh ra: `report/evidence/e0..e3.json`, `report/evidence/e4*.json`, ảnh minh hoạ trong
   `report/figs/`, và `/kaggle/working/day10_report.zip` để tải về commit vào repo.

> `lab_day10.ipynb` là khung Colab gốc; đây là bản chạy được trên Kaggle.

## 0. Môi trường (Kaggle GPU)

In [ ]:
# Kiểm tra GPU (Settings → Accelerator → GPU T4 x2 / P100)
!nvidia-smi

In [ ]:
# Đưa repo lab vào /kaggle/working (repo cần có scripts/score.py, không sửa).
# Cách A: Add Input repo dạng dataset  -> notebook tự copy.
# Cách B: push repo lên GitHub rồi điền REPO_URL dưới đây.
import os, sys, shutil, subprocess
from pathlib import Path

REPO_URL = ""  # ví dụ: "https://github.com/<username>/K4-Track-4-Day10-VisionLanguageModel.git"

WORK = Path("/kaggle/working")
REPO_DIR = WORK / "K4-Track-4-Day10-VisionLanguageModel"


def _looks_like_repo(p: Path) -> bool:
    return (p / "scripts" / "score.py").exists() and (p / "README.md").exists()


if not _looks_like_repo(REPO_DIR):
    src = None
    for hit in Path("/kaggle/input").rglob("scripts/score.py"):
        root = hit.parent.parent
        if _looks_like_repo(root):
            src = root
            break
    if src is not None:
        print("Copy repo từ input:", src)
        shutil.copytree(src, REPO_DIR, dirs_exist_ok=True)
    elif REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        raise SystemExit("Không thấy repo. Hãy Add Input repo, hoặc điền REPO_URL.")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("CWD =", Path.cwd())
print(sorted(p.name for p in Path.cwd().iterdir()))

In [ ]:
# Cài đúng phiên bản (Kaggle đã có torch/torchvision). Nếu vừa cài xong, Restart Session rồi chạy lại.
!pip -q install -r requirements-colab.txt

In [ ]:
import itertools, json, math, random, re, subprocess, time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from scripts.labcommon import CLASSES, read_jsonl, write_jsonl

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
DATA = Path("data")
WORK_DIR = Path("work");   WORK_DIR.mkdir(exist_ok=True)
EVID = Path("report/evidence"); EVID.mkdir(parents=True, exist_ok=True)
FIGS = Path("report/figs");     FIGS.mkdir(parents=True, exist_ok=True)

print("CLASSES =", CLASSES)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Dữ liệu: tải và chia split theo phiên quay

In [ ]:
# 1) Kaggle secrets (tuỳ chọn, chỉ để dùng chế độ --kaggle)
try:
    from kaggle_secrets import UserSecretsClient
    _c = UserSecretsClient()
    for _k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        try:
            os.environ[_k] = _c.get_secret(_k)
        except Exception:
            pass
    print("Kaggle secrets: có")
except Exception as e:
    print("Kaggle secrets: không dùng được ->", e)

# 2) Tìm dataset đã Add Input
source = None
for p in Path("/kaggle/input").glob("*"):
    if "road" in p.name.lower() or "damage" in p.name.lower():
        source = str(p)
        break

# 3) Chia split (cố định theo phiên, tái lập được)
if source:
    print("Dùng --source", source)
    !python scripts/prepare_data.py --source {source}
elif os.environ.get("KAGGLE_USERNAME"):
    !python scripts/prepare_data.py --kaggle
else:
    raise SystemExit("Thêm dataset làm Input, hoặc cấu hình Kaggle secrets.")

In [ ]:
# Kiểm tra split: số ảnh/box, và KHÔNG phiên nào nằm ở hai split
meta = json.loads((DATA / "splits" / "meta.json").read_text(encoding="utf-8"))
print("dataset_version:", meta["dataset_version"])

train_entries = read_jsonl(DATA / "splits" / "train.jsonl")
val_entries = read_jsonl(DATA / "splits" / "val.jsonl")
test_entries = read_jsonl(DATA / "splits" / "test.jsonl")

sessions = {}
for name, rows in (("train", train_entries), ("val", val_entries), ("test", test_entries)):
    nbox = sum(len(r["objects"]) for r in rows)
    per = {c: sum(1 for r in rows for o in r["objects"] if o["label"] == c) for c in CLASSES}
    print(f"{name:5s} images={len(rows):4d} boxes={nbox:5d} {per}")
    for r in rows:
        sessions.setdefault(r["session"], set()).add(name)

leak = {k: v for k, v in sessions.items() if len(v) > 1}
assert not leak, f"RÒ RỈ PHIÊN: {leak}"
print("OK: không phiên nào ở hai split")

In [ ]:
def load_image(entry, root=DATA):
    return Image.open(root / entry["image_path"]).convert("RGB")


def yolo_to_pixel(o, w, h):
    xc, yc, bw, bh = o["yolo"]
    return ((xc - bw / 2) * w, (yc - bh / 2) * h, (xc + bw / 2) * w, (yc + bh / 2) * h)


COLORS = {"pothole": "red", "crack": "lime", "manhole": "deepskyblue"}


fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, e in zip(axes.ravel(), train_entries[:6]):
    ax.imshow(np.asarray(load_image(e))); ax.axis("off")
    for o in e["objects"]:
        x1, y1, x2, y2 = yolo_to_pixel(o, e["width"], e["height"])
        c = COLORS.get(o["label"], "yellow")
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=c, lw=2))
        ax.text(x1, max(0, y1 - 3), o["label"], color=c, fontsize=7)
    ax.set_title(e["image_id"], fontsize=7)
plt.tight_layout(); plt.savefig(FIGS / "gt_samples.png", dpi=120); plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
PROMPT = (
    "Bạn là hệ thống phát hiện hư hại mặt đường từ ảnh dashcam.\n"
    "Chỉ trả về DUY NHẤT một JSON list, không thêm chữ nào khác. Mỗi phần tử có dạng:\n"
    '{"label": "<pothole|crack|manhole>", "bbox_2d": [x1, y1, x2, y2]}\n'
    "Toạ độ là số nguyên trong thang 0-1000, tương đối theo chiều rộng và chiều cao ảnh "
    "(x1,y1 là góc trên-trái; x2,y2 là góc dưới-phải).\n"
    "Nếu ảnh không có hư hại nào, trả về []."
)
PROMPT_PIXEL = (
    "Bạn là hệ thống phát hiện hư hại mặt đường từ ảnh dashcam.\n"
    "Chỉ trả về DUY NHẤT một JSON list. Mỗi phần tử có dạng:\n"
    '{"label": "<pothole|crack|manhole>", "bbox_2d": [x1, y1, x2, y2]}\n'
    "Toạ độ là số nguyên theo PIXEL của ảnh gốc (góc trên-trái và góc dưới-phải).\n"
    "Nếu ảnh không có hư hại nào, trả về []."
)


def _clamp1000(v):
    return max(0, min(1000, int(round(v))))


def yolo_to_box(label, yolo):
    xc, yc, w, h = yolo
    return {"label": label, "bbox_2d": [
        _clamp1000((xc - w / 2) * 1000), _clamp1000((yc - h / 2) * 1000),
        _clamp1000((xc + w / 2) * 1000), _clamp1000((yc + h / 2) * 1000)]}


def yolo_to_box_pixel(label, yolo, width, height):
    xc, yc, w, h = yolo
    cl = lambda v, m: max(0, min(m, int(round(v))))
    return {"label": label, "bbox_2d": [
        cl((xc - w / 2) * width, width), cl((yc - h / 2) * height, height),
        cl((xc + w / 2) * width, width), cl((yc + h / 2) * height, height)]}


def objects_to_answer(objects, coord_scale="norm1000", width=None, height=None):
    if coord_scale == "pixel":
        boxes = [yolo_to_box_pixel(o["label"], o["yolo"], width, height) for o in objects]
    else:
        boxes = [yolo_to_box(o["label"], o["yolo"]) for o in objects]
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))  # trên->dưới, trái->phải
    return json.dumps(boxes, separators=(",", ":"), ensure_ascii=False)

In [ ]:
# Quy ngược JSON -> pixel và vẽ chồng box gốc: phải TRÙNG
e = train_entries[0]
ans = objects_to_answer(e["objects"])
print("answer:", ans[:200])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.imshow(np.asarray(load_image(e))); ax.axis("off")
for b in json.loads(ans):
    x1 = b["bbox_2d"][0] * e["width"] / 1000; y1 = b["bbox_2d"][1] * e["height"] / 1000
    x2 = b["bbox_2d"][2] * e["width"] / 1000; y2 = b["bbox_2d"][3] * e["height"] / 1000
    ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="yellow", lw=1.5, ls="--"))
for o in e["objects"]:
    x1, y1, x2, y2 = yolo_to_pixel(o, e["width"], e["height"])
    ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="red", lw=1.5))
plt.title("đỏ = GT, vàng = quy ngược từ JSON"); plt.show()

## 3. E0: zero-shot

In [ ]:
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration


def load_processor(model_id=MODEL_ID):
    proc = AutoProcessor.from_pretrained(model_id)
    proc.tokenizer.padding_side = "left"
    return proc


def load_base_model(model_id=MODEL_ID):
    quant = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        llm_int8_skip_modules=["visual", "lm_head"])  # giữ ViT + lm_head ở fp16
    return Qwen3VLForConditionalGeneration.from_pretrained(
        model_id, quantization_config=quant, dtype=torch.float16, device_map={"": 0})


def chat_inputs(processor, conversations, add_generation_prompt):
    return processor.apply_chat_template(
        conversations, tokenize=True, add_generation_prompt=add_generation_prompt,
        return_dict=True, return_tensors="pt", padding=True)


def make_messages(images, prompts):
    return [[{"role": "user", "content": [{"type": "image", "image": img},
                                          {"type": "text", "text": p}]}]
            for img, p in zip(images, prompts)]


@torch.no_grad()
def run_inference(model, processor, entries, out_path, batch_size=4,
                  max_new_tokens=256, coord_scale="norm1000"):
    model.eval()
    prompt = PROMPT_PIXEL if coord_scale == "pixel" else PROMPT
    rows = []
    for i in tqdm(range(0, len(entries), batch_size)):
        batch = entries[i:i + batch_size]
        inputs = chat_inputs(processor, make_messages([load_image(e) for e in batch],
                                                      [prompt] * len(batch)), True)
        inputs = inputs.to(model.device)
        prompt_len = inputs["input_ids"].shape[1]
        gen = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        for j, e in enumerate(batch):
            raw = processor.tokenizer.decode(gen[j, prompt_len:], skip_special_tokens=True)
            rows.append({"image_id": e["image_id"], "raw": raw})
    write_jsonl(out_path, rows)
    print("wrote", out_path, len(rows), "dòng")
    return rows

In [ ]:
processor = load_processor()
model = load_base_model()

e0_rows = run_inference(model, processor, test_entries, WORK_DIR / "e0.jsonl")
for r in e0_rows[:5]:
    print("-", r["image_id"], "->", r["raw"][:200].replace("\n", " "))

del model
torch.cuda.empty_cache()

In [ ]:
# Chấm E0 (trọng tài của đề)
subprocess.run(["python", "scripts/score.py", "--pred", "work/e0.jsonl",
                "--name", "E0", "--out", "report/evidence/e0.json"], check=True)

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
def encode_example(entry, processor, coord_scale="norm1000"):
    img = load_image(entry)
    answer = objects_to_answer(entry["objects"], coord_scale, entry["width"], entry["height"])
    prompt = PROMPT_PIXEL if coord_scale == "pixel" else PROMPT
    user = {"role": "user", "content": [{"type": "image", "image": img},
                                        {"type": "text", "text": prompt}]}
    asst = {"role": "assistant", "content": [{"type": "text", "text": answer}]}

    prompt_enc = chat_inputs(processor, [[user]], add_generation_prompt=True)
    full_enc = chat_inputs(processor, [[user, asst]], add_generation_prompt=False)

    n = prompt_enc["input_ids"].shape[1]
    assert torch.equal(full_enc["input_ids"][0, :n], prompt_enc["input_ids"][0]), \
        "prompt phải là tiền tố của chuỗi đầy đủ"

    labels = full_enc["input_ids"].clone()
    labels[:, :n] = -100                       # chỉ học phần trả lời

    item = {"input_ids": full_enc["input_ids"][0],
            "attention_mask": full_enc["attention_mask"][0],
            "labels": labels[0],
            "pixel_values": full_enc["pixel_values"],
            "image_grid_thw": full_enc["image_grid_thw"]}
    if "mm_token_type_ids" in full_enc:
        item["mm_token_type_ids"] = full_enc["mm_token_type_ids"][0]
    return item, answer


def make_collate(processor):
    pad_id = processor.tokenizer.pad_token_id or 0
    pad_vals = {"input_ids": pad_id, "attention_mask": 0, "mm_token_type_ids": 0, "labels": -100}

    def collate(batch):
        keys = [k for k in ("input_ids", "attention_mask", "mm_token_type_ids", "labels")
                if k in batch[0]]
        maxlen = max(b["input_ids"].shape[0] for b in batch)
        out = {}
        for k in keys:
            seqs = []
            for b in batch:
                t = b[k]; pad = maxlen - t.shape[0]
                if pad > 0:
                    t = torch.cat([t, torch.full((pad,), pad_vals[k], dtype=t.dtype)])
                seqs.append(t)
            out[k] = torch.stack(seqs)
        out["pixel_values"] = torch.cat([b["pixel_values"] for b in batch], dim=0)
        out["image_grid_thw"] = torch.cat([b["image_grid_thw"] for b in batch], dim=0)
        return out

    return collate

In [ ]:
# Kiểm tra: decode riêng phần labels != -100 phải đúng JSON đáp án (+ <|im_end|>)
item, answer = encode_example(train_entries[0], processor)
print("answer:", answer[:200])
print("độ dài chuỗi:", item["input_ids"].shape[0])
if "mm_token_type_ids" in item:
    print("số token ảnh (mm==1):", int((item["mm_token_type_ids"] == 1).sum()))
img_tok = getattr(processor, "image_token_id", None)
if img_tok is not None:
    print("số token image_pad:", int((item["input_ids"] == img_tok).sum()))
print("labels != -100:", processor.tokenizer.decode(
    [t for t in item["labels"].tolist() if t != -100], skip_special_tokens=False)[:250])

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
from peft import LoraConfig, get_peft_model

LLM_RE = (r"model\.language_model\.layers\.\d+\.self_attn\.(q|k|v|o)_proj"
          r"|model\.language_model\.layers\.\d+\.mlp\.(gate|up|down)_proj")
VIT_RE = (r"model\.visual\.blocks\.\d+\.attn\.(qkv|proj)"
          r"|model\.visual\.blocks\.\d+\.mlp\.(linear_fc1|linear_fc2)")
TARGETS = {"E1": LLM_RE, "E2": LLM_RE, "E3": LLM_RE + "|" + VIT_RE}


def count_targets(model, pattern):
    rx = re.compile(pattern)
    return [n for n, _ in model.named_modules() if rx.fullmatch(n)]


def apply_lora(model, pattern, train_projector, r=16, alpha=32, tag="E?"):
    for p in model.parameters():
        p.requires_grad = False
    model.enable_input_require_grads()               # cần cho gradient checkpointing

    matched = count_targets(model, pattern)
    print(f"[{tag}] matched modules = {len(matched)}")
    cfg = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=0.05,
                     target_modules=pattern, bias="none", task_type="CAUSAL_LM")
    model = get_peft_model(model, cfg)

    # Xác nhận PEFT THỰC SỰ gắn LoRA (regex không khớp -> huấn luyện rỗng, kết quả vô nghĩa)
    injected = [n for n, _ in model.named_modules() if n.endswith(".lora_A")]
    print(f"[{tag}] LoRA modules injected = {len(injected)} (mong đợi {len(matched)})")
    if len(injected) != len(matched):
        raise RuntimeError(
            f"[{tag}] PEFT gắn {len(injected)} module nhưng regex khớp {len(matched)}; "
            "kiểm tra target_modules và phiên bản PEFT trước khi chạy.")

    if train_projector:
        n_proj = 0
        for name, p in model.named_parameters():
            if "visual.merger" in name or "visual.deepstack_merger_list" in name:
                p.data = p.data.to(torch.float32)     # projector học ở float32
                p.requires_grad = True
                n_proj += p.numel()
        print(f"[{tag}] projector trainable ≈ {n_proj/1e6:.1f}M")

    ntrain = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if ntrain == 0:
        raise RuntimeError(f"[{tag}] không có tham số nào được huấn luyện; dừng để tránh kết quả vô nghĩa.")
    print(f"[{tag}] trainable params = {ntrain/1e6:.1f}M")
    return model

In [ ]:
# Xác nhận tên module thật + số module khớp (E1/E2 = 252, E3 = 348)
_probe = load_base_model()
for s in ("E1", "E2", "E3"):
    print(s, "->", len(count_targets(_probe, TARGETS[s])), "module")
names = [n for n, _ in _probe.named_modules()]
print("ví dụ LLM:", [n for n in names if "language_model.layers.0." in n][:4])
print("ví dụ ViT:", [n for n in names if "visual.blocks.0." in n][:4])
del _probe
torch.cuda.empty_cache()

## 6. Vòng huấn luyện

In [ ]:
class RoadDamageDataset(Dataset):
    def __init__(self, entries, processor, coord_scale="norm1000", train_fraction=1.0, seed=42):
        self.entries = list(entries)
        if train_fraction < 1.0:
            rng = random.Random(seed)
            self.entries = rng.sample(self.entries, max(1, int(len(self.entries) * train_fraction)))
        self.processor = processor
        self.coord_scale = coord_scale

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, i):
        item, _ = encode_example(self.entries[i], self.processor, self.coord_scale)
        return item


def _proj_names(model):
    # chỉ các tham số projector ĐANG được huấn luyện (E1 đóng băng -> không lưu)
    return [n for n, p in model.named_parameters()
            if p.requires_grad and ("visual.merger" in n or "visual.deepstack_merger_list" in n)]


def save_extra(model, path):
    names = set(_proj_names(model))
    if not names:
        return 0
    sd = {n: p.detach().to(torch.float16).cpu()
          for n, p in model.named_parameters() if n in names}
    torch.save(sd, path)
    return len(sd)


def load_extra(model, path):
    if not Path(path).exists():
        print("không có extra_trainable:", path)
        return
    sd = torch.load(path, map_location="cpu")
    missing, unexpected = model.load_state_dict(sd, strict=False)
    print(f"nạp projector: {len(sd)} tensor (missing={len(missing)}, unexpected={len(unexpected)})")


def train_setup(tag, pattern, train_projector, out_dir, r=16, alpha=32, train_fraction=1.0,
                coord_scale="norm1000", lr=2e-4, batch_size=2, grad_accum=4, max_steps=300,
                seed=42, entries=None, log_every=50):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)

    processor = load_processor()
    model = load_base_model()
    model = apply_lora(model, pattern, train_projector, r=r, alpha=alpha, tag=tag)
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.config.use_cache = False

    ds = RoadDamageDataset(entries if entries is not None else train_entries,
                           processor, coord_scale, train_fraction, seed)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, collate_fn=make_collate(processor),
                    num_workers=2, drop_last=True, pin_memory=True)

    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    warmup = max(1, int(0.05 * max_steps))

    def lr_lambda(step):
        if step < warmup:
            return (step + 1) / warmup
        prog = (step - warmup) / max(1, max_steps - warmup)
        return 0.5 * (1 + math.cos(math.pi * prog))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    scaler = torch.amp.GradScaler("cuda")

    model.train()
    it = itertools.cycle(dl)
    t0 = time.time(); torch.cuda.reset_peak_memory_stats()
    losses = []
    for step in range(1, max_steps + 1):
        for _ in range(grad_accum):
            batch = next(it)
            batch = {k: v.to(model.device) for k, v in batch.items()}
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(**batch).loss / grad_accum
            if not torch.isfinite(loss):
                raise RuntimeError(f"loss không hữu hạn ở step {step}: {loss.item()}")
            scaler.scale(loss).backward()
            losses.append(float(loss.detach()) * grad_accum)
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)

        if step % log_every == 0:
            print(f"[{tag}] step {step}/{max_steps} loss={np.mean(losses[-log_every*grad_accum:]):.4f}")
        if step % 100 == 0:
            model.save_pretrained(out_dir); save_extra(model, out_dir / "extra_trainable.pt")

    model.save_pretrained(out_dir); save_extra(model, out_dir / "extra_trainable.pt")
    processor.save_pretrained(out_dir)

    run = {"tag": tag, "trainable_params": int(sum(p.numel() for p in model.parameters() if p.requires_grad)),
           "peak_vram_gb": torch.cuda.max_memory_allocated() / 1e9,
           "train_minutes": (time.time() - t0) / 60, "losses": losses, "r": r, "alpha": alpha,
           "train_fraction": train_fraction, "coord_scale": coord_scale, "max_steps": max_steps}
    (out_dir / "run.json").write_text(json.dumps(run, indent=2), encoding="utf-8")
    print(f"[{tag}] trainable={run['trainable_params']/1e6:.1f}M "
          f"peak_vram={run['peak_vram_gb']:.2f}GB time={run['train_minutes']:.1f}min")

    del model, opt
    torch.cuda.empty_cache()
    return run

In [ ]:
# E1: LoRA chỉ trên LLM; ViT và projector đóng băng
RUNS = {}
RUNS["E1"] = train_setup("E1", TARGETS["E1"], False, "work/adapters/e1")

## 7. Suy luận và chấm E1–E3

In [ ]:
def evaluate_setup(key, adapter_dir, extra_path=None, meta=None, coord_scale="norm1000",
                   entries=None, batch_size=4):
    entries = entries if entries is not None else test_entries
    processor = load_processor()
    base = load_base_model()
    if adapter_dir and Path(adapter_dir).exists():
        from peft import PeftModel
        model = PeftModel.from_pretrained(base, adapter_dir)
    else:
        model = base
    if extra_path:
        load_extra(model, extra_path)          # BẮT BUỘC cho E2/E3
    out = WORK_DIR / f"{key.lower()}.jsonl"
    run_inference(model, processor, entries, out, batch_size=batch_size, coord_scale=coord_scale)
    del model
    torch.cuda.empty_cache()

    cmd = ["python", "scripts/score.py", "--pred", str(out), "--name", key,
           "--out", f"report/evidence/{key.lower()}.json", "--coord", coord_scale]
    for k, v in (meta or {}).items():
        cmd += ["--meta", f"{k}={v}"]
    subprocess.run(cmd, check=True)


def meta_from(run):
    return {"trainable_params": run["trainable_params"],
            "peak_vram_gb": round(run["peak_vram_gb"], 3),
            "train_minutes": round(run["train_minutes"], 2)}

In [ ]:
evaluate_setup("E1", "work/adapters/e1", meta=meta_from(RUNS["E1"]))

In [ ]:
# E2: E1 + mở đầy đủ projector
RUNS["E2"] = train_setup("E2", TARGETS["E2"], True, "work/adapters/e2")
evaluate_setup("E2", "work/adapters/e2", extra_path="work/adapters/e2/extra_trainable.pt",
               meta=meta_from(RUNS["E2"]))

In [ ]:
# E3: E2 + LoRA trên ViT
RUNS["E3"] = train_setup("E3", TARGETS["E3"], True, "work/adapters/e3")
evaluate_setup("E3", "work/adapters/e3", extra_path="work/adapters/e3/extra_trainable.pt",
               meta=meta_from(RUNS["E3"]))

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
def best_setup():
    best, bf1 = None, -1.0
    for k in ("E1", "E2", "E3"):
        p = EVID / f"{k.lower()}.json"
        if p.exists():
            f = json.loads(p.read_text(encoding="utf-8"))["micro"]["f1"]
            if f > bf1:
                best, bf1 = k, f
    return best


BEST = best_setup()
print("setup tốt nhất:", BEST)

In [ ]:
# E4 chọn biến "rank": r = 4 và r = 64 (alpha = 2*r), mọi thứ khác giữ như BEST.
E4 = {}
for r in (4, 64):
    tag = f"E4rank{r}"
    run = train_setup(tag, TARGETS[BEST], BEST in ("E2", "E3"),
                      f"work/adapters/e4_r{r}", r=r, alpha=2 * r)
    extra = f"work/adapters/e4_r{r}/extra_trainable.pt" if BEST in ("E2", "E3") else None
    evaluate_setup(tag, f"work/adapters/e4_r{r}", extra_path=extra, meta=meta_from(run))
    E4[r] = run

In [ ]:
# E4 (tuỳ chọn) hai biến khác — chỉ chạy khi cần thêm bằng chứng.
RUN_OTHER_E4 = False
if RUN_OTHER_E4:
    # 1) coord_scale: norm1000 (đã có) vs pixel
    run_px = train_setup("E4coordpixel", TARGETS[BEST], BEST in ("E2", "E3"),
                         "work/adapters/e4_coordpixel", coord_scale="pixel")
    extra = "work/adapters/e4_coordpixel/extra_trainable.pt" if BEST in ("E2", "E3") else None
    evaluate_setup("E4coordpixel", "work/adapters/e4_coordpixel", extra_path=extra,
                   coord_scale="pixel", meta=meta_from(run_px))

    # 2) data_size: 10% và 100% ảnh train
    for frac in (0.1, 1.0):
        tag = f"E4data{int(frac*100)}"
        run = train_setup(tag, TARGETS[BEST], BEST in ("E2", "E3"),
                          f"work/adapters/e4_data{int(frac*100)}", train_fraction=frac)
        ex = f"work/adapters/e4_data{int(frac*100)}/extra_trainable.pt" if BEST in ("E2", "E3") else None
        evaluate_setup(tag, f"work/adapters/e4_data{int(frac*100)}", extra_path=ex, meta=meta_from(run))

In [ ]:
# E5 reward IoU — hàm Python thuần, TEST ĐƯỢC KHÔNG CẦN GPU
from scripts.score import parse_output, iou_xyxy


def reward_iou(completion, gt_boxes, iou_thr=0.5):
    # completion: str output model; gt_boxes: [{"label","bbox"}] theo thang 0-1000 (x1,y1,x2,y2)
    pred = parse_output(completion)
    if pred is None:
        return 0.0
    if not gt_boxes and not pred:
        return 1.0
    pairs = []
    for gi, g in enumerate(gt_boxes):
        for pi, p in enumerate(pred):
            if g["label"] != p["label"]:
                continue
            v = iou_xyxy(g["bbox"], p["bbox"])
            if v >= iou_thr:
                pairs.append((v, gi, pi))
    pairs.sort(reverse=True)
    ug, up, total = set(), set(), 0.0
    for v, gi, pi in pairs:
        if gi in ug or pi in up:
            continue
        ug.add(gi); up.add(pi); total += v
    return 0.2 + 0.8 * total / max(len(gt_boxes), len(pred), 1)


def _box(l, x1, y1, x2, y2):
    return {"label": l, "bbox_2d": [x1, y1, x2, y2]}


G = [{"label": "pothole", "bbox": (400, 400, 600, 600)}]
assert reward_iou(json.dumps([_box("pothole", 400, 400, 600, 600)]), G) == 1.0       # hoàn hảo
assert reward_iou("garbage", G) == 0.0                                               # JSON hỏng
assert abs(reward_iou(json.dumps([_box("crack", 400, 400, 600, 600)]), G) - 0.2) < 1e-9   # sai lớp
assert abs(reward_iou(json.dumps([]), G) - 0.2) < 1e-9                               # rỗng
assert reward_iou(json.dumps([]), []) == 1.0                                         # cùng rỗng
extra = reward_iou(json.dumps([_box("pothole", 400, 400, 600, 600),
                               _box("pothole", 1, 1, 50, 50)]), G)
assert abs(extra - (0.2 + 0.8 * 1.0 / 2)) < 1e-9                                     # thừa box bị phạt
print("reward tests OK")

In [ ]:
# E5 GRPO (tuỳ chọn, +10). Bật khi đã xong E0–E4. API trl phụ thuộc phiên bản (xem requirements).
RUN_GRPO = False
if RUN_GRPO:
    from datasets import Dataset as HFDataset, Features, Image as HFImage, Value
    from peft import PeftModel
    from trl import GRPOConfig, GRPOTrainer

    ADAPTER = {"E1": "work/adapters/e1", "E2": "work/adapters/e2", "E3": "work/adapters/e3"}[BEST]

    def gt_of(entry):
        out = []
        for o in entry["objects"]:
            xc, yc, w, h = o["yolo"]
            out.append({"label": o["label"], "bbox": ((xc - w / 2) * 1000, (yc - h / 2) * 1000,
                                                      (xc + w / 2) * 1000, (yc + h / 2) * 1000)})
        return out

    # Lưu ảnh ra đĩa + khai báo feature Image để Dataset tạo được (tránh lỗi cast PIL).
    img_dir = Path("work/grpo_images"); img_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for e in train_entries:
        p = img_dir / f"{e['image_id']}.png"
        load_image(e).save(p)
        rows.append({"prompt": PROMPT, "image": str(p), "gt_json": json.dumps(gt_of(e))})
    hf_ds = HFDataset.from_list(
        rows, features=Features({"prompt": Value("string"), "image": HFImage(),
                                 "gt_json": Value("string")}))

    def reward_fn(completions, gt_json=None, **kwargs):
        return [reward_iou(c, json.loads(g)) for c, g in zip(completions, gt_json)]

    proc = load_processor()
    base = load_base_model()
    model = PeftModel.from_pretrained(base, ADAPTER)
    ex = Path(ADAPTER) / "extra_trainable.pt"
    if BEST in ("E2", "E3") and ex.exists():
        load_extra(model, ex)

    cfg = GRPOConfig(output_dir="work/grpo", num_generations=4, max_completion_length=128,
                     max_steps=60, learning_rate=5e-6, beta=0.0, use_vllm=False,
                     per_device_train_batch_size=1, gradient_accumulation_steps=4,
                     logging_steps=1, remove_unused_columns=False)
    trainer = GRPOTrainer(model=model, args=cfg, train_dataset=hf_ds, reward_funcs=reward_fn)
    trainer.train()
    trainer.save_model("work/adapters/e5_grpo")

## Tổng hợp kết quả & đóng gói nộp bài

In [ ]:
import glob
import pandas as pd
from IPython.display import display

rows = []
for f in sorted(glob.glob("report/evidence/e*.json")):
    d = json.loads(Path(f).read_text(encoding="utf-8"))
    m, pc, mt = d["micro"], d["per_class"], d.get("meta", {})
    rows.append([d["name"], mt.get("trainable_params", ""), mt.get("peak_vram_gb", ""),
                 mt.get("train_minutes", ""), round(m["f1"], 3),
                 f'{d["f1_ci95"][0]:.3f}-{d["f1_ci95"][1]:.3f}', round(d["fp_per_image"], 2),
                 round(pc["pothole"]["f1"], 3), round(pc["crack"]["f1"], 3),
                 round(pc["manhole"]["f1"], 3)])
df = pd.DataFrame(rows, columns=["Mã", "tham số", "VRAM(GB)", "phút", "F1", "CI95", "FP/ảnh",
                                 "F1 pothole", "F1 crack", "F1 manhole"])
display(df)
df.to_csv("report/summary.csv", index=False)

In [ ]:
# Ảnh minh hoạ ≥6 dự đoán (đúng/sai) của setup tốt nhất: box đỏ = GT, box xanh = model
from scripts.score import parse_output

adapter = {"E1": "work/adapters/e1", "E2": "work/adapters/e2", "E3": "work/adapters/e3"}[BEST]
extra = Path(adapter) / "extra_trainable.pt"
proc = load_processor()
base = load_base_model()
from peft import PeftModel
model = PeftModel.from_pretrained(base, adapter)
if BEST in ("E2", "E3") and extra.exists():
    load_extra(model, extra)

sample = test_entries[20:26]
preds = run_inference(model, proc, sample, WORK_DIR / "best_preview.jsonl")
del model
torch.cuda.empty_cache()
raw_by_id = {r["image_id"]: r["raw"] for r in preds}

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, e in zip(axes.ravel(), sample):
    ax.imshow(np.asarray(load_image(e))); ax.axis("off")
    for o in e["objects"]:
        x1, y1, x2, y2 = yolo_to_pixel(o, e["width"], e["height"])
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="red", lw=2))
    parsed = parse_output(raw_by_id.get(e["image_id"], "")) or []
    for b in parsed:
        bb = b["bbox"]
        x1 = bb[0] * e["width"] / 1000; y1 = bb[1] * e["height"] / 1000
        x2 = bb[2] * e["width"] / 1000; y2 = bb[3] * e["height"] / 1000
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="blue", lw=2, ls="--"))
    ax.set_title(e["image_id"] + f" | pred={len(parsed)} gt={len(e['objects'])}", fontsize=7)
plt.tight_layout(); plt.savefig(FIGS / "best_predictions.png", dpi=120); plt.show()
print("đỏ = GT, xanh = dự đoán")

In [ ]:
# Đóng gói report + evidence + notebook để tải về commit (KHÔNG commit data/ work/ trọng số)
shutil.make_archive("/kaggle/working/day10_report", "zip", root_dir=".", base_dir="report")
try:
    shutil.copy("kaggle_day10.ipynb", "report/kaggle_day10.ipynb")
except Exception as exc:
    print("không copy được notebook:", exc)
print("Đã tạo /kaggle/working/day10_report.zip")
print("Nhớ: điền REPORT.md bằng số của bạn,>=5 commit, rồi chạy python scripts/check_submission.py")